# 15 - Runtime-Assurance and Control-Barrier-Function Benchmark

**Purpose.** Comparative evaluation of CRI-supervised activation against two established safety-supervision
paradigms, under identical plant, candidate controller (OFO), fallback controller (droop), scenarios S1-S8, seeds,
horizon, sampling interval and operational limits:

* **RTA** - Simplex-style runtime assurance (model-predictive recoverability monitor + switch to the existing fallback).
* **CBF** - discrete-time control-barrier-function safety filter (CBF-QP) on the candidate command.

**Information parity.** Both baselines see only what CRI sees (measured, possibly noisy outputs, applied command,
operational limits) **plus the nominal plant model** (the same nominal parameters the candidate OFO already embeds).
They never see hidden truth (actuator gain, topology, loads/PV, true noise level). No guarantee is claimed for the
true plant: the RTA/CBF properties hold only for the nominal model.

**No leakage.** RTA/CBF design parameters were fixed on development trials (S10 randomized generator, seeds 1000-1004),
which are disjoint from S1-S8 (Sec. 2). One fixed configuration is used for every scenario and every LOSO fold.
DC-CRI uses the per-fold parameters already stored in `results/results_dc_cri_loso_folds.csv` (each calibrated
without its held-out scenario, notebook 13); nothing is recalibrated here.

**Two protocols, never mixed.**
* Protocol A (EC protocol, as in notebooks 08-10): instantaneous mechanism-based ground truth $z_k^{\rm unsafe}$.
* Protocol B (LOSO protocol, as in notebook 13): finite-horizon hard-limit label $z_k^{(H)}$, $T_H=0.5$ s.

**Code.** `cri_microgrid/baselines_rta_cbf.py` (predictor, RTA, CBF), `cri_microgrid/benchmark_sim.py`
(closed-loop runner with a pluggable supervisor; bit-identical to `sim.run_closed_loop` for Direct/Hard-limit),
`cri_microgrid/benchmark_eval.py` (scenario builders, metrics, protocols). No other module is modified.

**Outputs.** `results/r1c7_*.csv`, `results/r1c7_table_*.tex` (LaTeX tables generated from the CSVs),
`results/r1c7_run_metadata.json`. Runtime about 10 min on a desktop CPU; the supervisor timing columns
(`sup_time_*_ms`) are machine dependent, all other values are deterministic.


In [1]:
import sys, os, time, json, platform, warnings, dataclasses, datetime
sys.path.insert(0, os.path.abspath(".."))
import numpy as np, pandas as pd, scipy
warnings.filterwarnings("ignore", message="Values in x were outside bounds")

from cri_microgrid.config import (PlantParams, ControllerParams, ConstraintParams, CRIParams, ProbeParams,
                                   ProfileParams, SimulationParams)
from cri_microgrid import scenarios, sim, metrics, validation, plotstyle
from cri_microgrid import dc_calibration as dcc
from cri_microgrid import benchmark_sim as bsim
from cri_microgrid import baselines_rta_cbf as bl
from cri_microgrid.benchmark_eval import (Bench, SCENARIO_ORDER, NOISY, H_PRINCIPAL, load_dc_fold_params,
                                          scenario_builders)

RESULTS_DIR = os.path.abspath("../results"); FIG_DIR = os.path.abspath("../Figures")
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
T_START = time.time()

RTA_PRM, CBF_PRM, PRED_PRM = bl.RTAParams(), bl.CBFParams(), bl.PredictorParams()
DEV_SEEDS = [1000, 1001, 1002, 1003, 1004]
SEED_REP = 1                                    # representative seed
SEEDS_A_NOISY = list(range(20))                 # Protocol A: 20-seed study for the noisy scenarios S6, S8
LOSO_EVAL_SEEDS = {s: ([0, 1, 2, 3, 4] if s in NOISY else [1]) for s in SCENARIO_ORDER}  # = notebook 13

meta = dict(
    created=datetime.datetime.now().isoformat(timespec="seconds"),
    python=platform.python_version(), numpy=np.__version__, scipy=scipy.__version__, pandas=pd.__version__,
    platform=platform.platform(), processor=platform.processor(),
    predictor=dataclasses.asdict(PRED_PRM), rta=dataclasses.asdict(RTA_PRM), cbf=dataclasses.asdict(CBF_PRM),
    dev_seeds=DEV_SEEDS, seed_rep=SEED_REP, seeds_A_noisy=SEEDS_A_NOISY, loso_eval_seeds=LOSO_EVAL_SEEDS,
    H_principal=H_PRINCIPAL, Ts=PlantParams().Ts,
    dc_cri_fold_params="results/results_dc_cri_loso_folds.csv (unchanged, from notebook 13)",
    qp_solver="scipy.optimize.minimize(method='SLSQP')",
)
print(json.dumps(meta, indent=1))

{
 "created": "2026-11-19T23:02:03",
 "python": "3.10.9",
 "numpy": "2.2.6",
 "scipy": "1.15.3",
 "pandas": "2.3.3",
 "platform": "Windows-10-10.0.19045-SP0",
 "processor": "AMD64 Family 25 Model 80 Stepping 0, AuthenticAMD",
 "predictor": {
  "alpha_y": 0.5,
  "alpha_d": 0.1
 },
 "rta": {
  "horizon_steps": 25,
  "return_dwell_steps": 100,
  "tightening": {
   "delta_V": 0.005,
   "delta_dc": 0.008,
   "delta_I": 0.02
  }
 },
 "cbf": {
  "gamma": 1.0,
  "feas_tol": 1e-08,
  "tightening": {
   "delta_V": 0.005,
   "delta_dc": 0.008,
   "delta_I": 0.02
  },
  "modified_tol": 0.001,
  "maxiter": 100,
  "ftol": 1e-10
 },
 "dev_seeds": [
  1000,
  1001,
  1002,
  1003,
  1004
 ],
 "seed_rep": 1,
 "seeds_A_noisy": [
  0,
  1,
  2,
  3,
  4,
  5,
  6,
  7,
  8,
  9,
  10,
  11,
  12,
  13,
  14,
  15,
  16,
  17,
  18,
  19
 ],
 "loso_eval_seeds": {
  "S1": [
   1
  ],
  "S2": [
   1
  ],
  "S3": [
   1
  ],
  "S4": [
   1
  ],
  "S5": [
   1
  ],
  "S6": [
   0,
   1,
   2,
   3,
   4
  ],


## 1. Baseline definitions

**Shared nominal predictor** (structure of `plant.HybridACDCPlant.step` with unity gain, nominal topology, both VSCs online):
$p_{k+1}=p_k+A(\mathrm{sat}(u_k)-p_k)$, $V_{k+1}=V_k+a_v(w+Mp_k-V_k)$,
$E_{k+1}=E_k+T_s(d-(p_{1,k+1}+p_{3,k+1})/\eta)$ with $E=C_{dc}V_{dc}^2/2$, $I_{i,k+1}=\|(P_i,Q_i)_{k+1}\|/V_{i,k+1}$.
Lumped disturbances $w$ (AC) and $d$ (DC balance) are re-estimated each step by inverting the model on successive
(EMA-filtered, $\alpha_y$) measurements and smoothed with $\alpha_d$; they are held constant over a prediction.

**A. Simplex RTA.** Monitor: predict $N_p=25$ steps (0.5 s) applying $u_k^{\rm cand}$ for one step and the fallback law
$\pi_{\rm safe}(\hat y)$ thereafter; pass iff all predicted $V_1,V_2,V_{dc},I_1,I_2$ stay inside the tightened set
($\delta_V=0.005$, $\delta_{dc}=0.008$, $\delta_I=0.02$ pu). Switch $1\to0$ on the first failed check; return $0\to1$
after $M_{\rm ret}$ consecutive passes. Authority is binary ($\kappa=1$); the candidate keeps its own (unscaled) update,
exactly as under Direct/Hard-limit.

**B. Discrete-time CBF-QP.** Barriers $h_{dc}^{lo}=E-E(V_{dc,\min}^t)$, $h_{dc}^{hi}=E(V_{dc,\max}^t)-E$ (relative degree 1),
$h_{V}^{lo/hi}$ on the one-step-shifted voltage $V_{k+1}$ (relative degree 2, discrete high-order form), $h_I=I_{\max}^t-I$
(second-order cone). Filter: $u_k^*=\arg\min_u\|D_u^{-1}(u-u_k^{\rm cand})\|^2$ s.t. $h(x_{k+1}(u))\ge(1-\gamma)h(x_k)$ and the
setpoint box. If infeasible: minimal uniform relaxation $t^*$ of all rows, then the QP with rows relaxed by $t^*$.
The candidate is nominally always active; the **unmodified-command indicator** $s_k^{\rm unf}=\mathbb I(\|u_k^*-u_k^{\rm cand}\|_\infty\le10^{-3})$
is used for $A_c$, FAR, FDR and $N_{FA}$ of CBF. Filtered steps are therefore **not** counted as candidate authority.

## 2. Development-set selection (S10 randomized trials, seeds 1000-1004; disjoint from S1-S8)

In [2]:
pp, cp, cons, crip, probep, prof = PlantParams(), ControllerParams(), ConstraintParams(), CRIParams(), ProbeParams(), ProfileParams()
def dev_scenario(sd): return scenarios.build_randomized_trial(np.random.default_rng(sd), pp, prof, T_end=40.0)

def prediction_error(lg, alpha_y, H=25):
    pr = bl.NominalMicrogridPredictor(pp, bl.PredictorParams(alpha_y=alpha_y, alpha_d=PRED_PRM.alpha_d))
    Y = np.c_[lg["V1"], lg["V2"], lg["Vdc"], lg["I1"], lg["I2"], lg["P_POI"], lg["Q_POI"]]
    U = np.c_[lg["P1_cmd"], lg["Q1_cmd"], lg["P2_cmd"], lg["Q2_cmd"]]
    Yt = np.c_[lg["V1_true"], lg["V2_true"], lg["Vdc_true"]]          # row k = true state at k+1
    u0 = np.array([cp.P_nom[0], cp.Q_nom[0], cp.P_nom[1], cp.Q_nom[1]])
    eH = []
    for k in range(len(Y) - H - 1):
        if k == 0: pr.init_actuator_state(u0); pr.observe(Y[0], None)
        else: pr.observe(Y[k], U[k - 1])
        if not pr.ready or k < 50: continue
        p, V, E, _ = pr.current_state()
        for j in range(H): p, V, E, _ = pr.step_model(p, V, E, U[k + j])
        eH.append(abs(pr.vdc_of(E) - Yt[k + H - 1, 2]))
    return float(np.percentile(eH, 95))

dev_rows = []
for sd in DEV_SEEDS:
    lgd = sim.run_closed_loop(dev_scenario(sd), pp, cp, cons, crip, probep, SimulationParams(T_end=40.0, seed=sd, strategy="direct"))
    for a in [1.0, 0.5, 0.3]:
        dev_rows.append(dict(param="alpha_y", value=a, seed=sd, criterion="p95 25-step Vdc error [pu]", score=prediction_error(lgd, a)))
for M in [25, 50, 100]:
    for sd in DEV_SEEDS:
        lg = bsim.run_closed_loop_supervised(dev_scenario(sd), pp, cp, cons, SimulationParams(T_end=40.0, seed=sd),
                                             bl.SimplexRTASupervisor(pp, cp, cons, bl.RTAParams(return_dwell_steps=M)))
        dev_rows.append(dict(param="RTA_M_ret", value=M, seed=sd, criterion="hard-limit violation samples",
                             score=int(dcc.hard_limit_violation(lg, cons).sum()), availability=lg["s"].mean()))
for g in [0.1, 0.2, 0.5, 1.0]:
    for sd in DEV_SEEDS:
        lg = bsim.run_closed_loop_supervised(dev_scenario(sd), pp, cp, cons, SimulationParams(T_end=40.0, seed=sd),
                                             bl.DiscreteCBFFilter(pp, cons, bl.CBFParams(gamma=g)))
        dev_rows.append(dict(param="CBF_gamma", value=g, seed=sd, criterion="hard-limit violation samples",
                             score=int(dcc.hard_limit_violation(lg, cons).sum()), availability=1 - np.nanmean(lg["diag_filtered"])))
dev = pd.DataFrame(dev_rows)
dev.to_csv(os.path.join(RESULTS_DIR, "r1c7_devset_selection.csv"), index=False)
sel_a = dev[dev.param == "alpha_y"].groupby("value").score.max()          # worst case over dev seeds
sel_M = dev[dev.param == "RTA_M_ret"].groupby("value").agg(total=("score", "sum"), avail=("availability", "mean"))
sel_g = dev[dev.param == "CBF_gamma"].groupby("value").agg(total=("score", "sum"), avail=("availability", "mean"))
print("alpha_y: worst-case p95 25-step Vdc error\n", sel_a.round(4))
print("RTA M_ret:\n", sel_M.round(3)); print("CBF gamma:\n", sel_g.round(3))
chosen = dict(alpha_y=float(sel_a.idxmin()),
              M_ret=int(sel_M.sort_values(["total", "avail"], ascending=[True, False]).index[0]),
              gamma=float(sel_g.sort_values(["total", "avail"], ascending=[True, False]).index[0]))
print("Selected:", chosen)
assert chosen == dict(alpha_y=PRED_PRM.alpha_y, M_ret=RTA_PRM.return_dwell_steps, gamma=CBF_PRM.gamma), \
    "Module defaults differ from the development-set selection"
print("Module defaults match the development-set selection.")

alpha_y: worst-case p95 25-step Vdc error
 value
0.3    0.0750
0.5    0.0511
1.0    0.0719
Name: score, dtype: float64
RTA M_ret:
        total  avail
value              
25.0   544.0  0.569
50.0   357.0  0.423
100.0  289.0  0.190
CBF gamma:
        total  avail
value              
0.1     21.0  0.680
0.2     21.0  0.803
0.5     24.0  0.894
1.0     13.0  0.905
Selected: {'alpha_y': 0.5, 'M_ret': 100, 'gamma': 1.0}
Module defaults match the development-set selection.


## 3. Validation checks

In [3]:
checks = []
def record(name, passed, detail):
    checks.append(dict(check=name, passed=bool(passed), detail=detail)); print(("PASS " if passed else "FAIL ") + name + " -- " + detail)

B = scenario_builders(pp, prof)
# 3.1 runner equivalence
keys = ["s", "V1", "V2", "Vdc", "I1", "I2", "V1_true", "V2_true", "Vdc_true", "P1_cmd", "Q1_cmd", "P2_cmd", "Q2_cmd", "P1_candidate"]
worst = 0.0
for sname in SCENARIO_ORDER:
    for strat, sup in [("direct", bsim.DirectSupervisor()), ("threshold", bsim.HardLimitSupervisor(cons))]:
        sc = B[sname](); a = sim.run_closed_loop(sc, pp, cp, cons, crip, probep, SimulationParams(T_end=sc.T_end, seed=SEED_REP, strategy=strat))
        sc = B[sname](); b = bsim.run_closed_loop_supervised(sc, pp, cp, cons, SimulationParams(T_end=sc.T_end, seed=SEED_REP), sup)
        worst = max(worst, max(float(np.max(np.abs(a[k] - b[k]))) for k in keys))
record("3.1 runner bit-identical to sim.run_closed_loop (Direct, Hard-limit; S1-S8)", worst == 0.0, f"max |diff| = {worst:g}")

# 3.2 nominal predictor reproduces the plant equations exactly when the plant is nominal (S1, clean, alpha=1)
sc = B["S1"](); lg = sim.run_closed_loop(sc, pp, cp, cons, crip, probep, SimulationParams(T_end=sc.T_end, seed=SEED_REP, strategy="direct"))
pr = bl.NominalMicrogridPredictor(pp, bl.PredictorParams(alpha_y=1.0, alpha_d=1.0))
Y = np.c_[lg["V1"], lg["V2"], lg["Vdc"], lg["I1"], lg["I2"], lg["P_POI"], lg["Q_POI"]]
U = np.c_[lg["P1_cmd"], lg["Q1_cmd"], lg["P2_cmd"], lg["Q2_cmd"]]
e1 = []
for k in range(len(Y) - 1):
    if k == 0: pr.init_actuator_state(np.array([cp.P_nom[0], 0, cp.P_nom[1], 0])); pr.observe(Y[0], None); continue
    pr.observe(Y[k], U[k - 1])
    if k < 200: continue                         # actuator-state estimate initialized at 0.29 vs true 0.30; decays
    p, V, E, _ = pr.current_state(); p, V, E, I = pr.step_model(p, V, E, U[k])
    e1.append(max(np.max(np.abs(V - [lg["V1_true"][k], lg["V2_true"][k]])), abs(pr.vdc_of(E) - lg["Vdc_true"][k])))
record("3.2 nominal predictor exact on nominal plant (S1, one-step, k>=200)", max(e1) < 1e-6, f"max one-step error = {max(e1):.2e} pu")

# 3.3 RTA logic: dwell and immediate switch-off
rta = bl.SimplexRTASupervisor(pp, cp, cons); rta.reset()
y_ss = Y[1000]; u_ss = U[999]; u_c = U[1000]
s_hist = [rta.step(k, k * pp.Ts, y_ss, u_ss, None, u_c)["s"] for k in range(RTA_PRM.return_dwell_steps + 5)]
first_on = int(np.argmax(np.array(s_hist) == 1))
record("3.3a RTA returns authority after exactly M_ret passing checks", first_on == RTA_PRM.return_dwell_steps,
       f"first s=1 at k={first_on} (predictor ready at k=1, M_ret={RTA_PRM.return_dwell_steps})")
# (i) switching rule: force one failed check while s=1 -> s must drop to 0 on that same step
_orig_check = rta._check
rta._check = lambda u: (False, -1.0, 1)
out = rta.step(len(s_hist), 0.0, y_ss, u_ss, None, u_c)
rta._check = _orig_check
record("3.3b RTA switches to fallback on the first failed check", s_hist[-1] == 1 and out["s"] == 0,
       f"s before={s_hist[-1]}, s after forced failure={out['s']}")
# (ii) monitor: one step of maximum export is recoverable from the healthy state (Simplex semantics),
#      but must be flagged when the DC link already sits 0.004 pu above the tightened lower limit
u_bad = np.array([1.0, 0.0, 1.0, 0.0])
ok_healthy, mm_healthy, _ = rta._check(u_bad)
rta.pred.yf = rta.pred.yf.copy(); rta.pred.yf[2] = cons.Vdc_min + RTA_PRM.tightening.delta_dc + 0.004
ok_low, mm_low, jf = rta._check(u_bad)
record("3.3c RTA monitor flags an unrecoverable command near the DC limit", (not ok_low) and mm_low < 0,
       f"healthy state: pass={ok_healthy} (margin {mm_healthy:.4f}); near-limit state: pass={ok_low}, first failing step={jf}, margin {mm_low:.4f}")

# 3.4 CBF solver: pass-through, analytic projection, infeasible relaxation
cbf = bl.DiscreteCBFFilter(pp, cons); cbf.reset()
for k in range(3): o = cbf.step(k, 0, y_ss, u_ss, None, u_c)
record("3.4a CBF passes a safe candidate unchanged", o["filtered"] == 0 and o["cbf_status"] == 0, f"dev={o['filter_dev_inf']:.2e}, status={o['cbf_status']}")
# one binding linear row: put the DC link just above the tightened lower limit and request strong export
pr = cbf.pred; pr.yf = pr.yf.copy(); pr.yf[2] = cons.Vdc_min + CBF_PRM.tightening.delta_dc + 0.004
u_exp = np.array([0.55, 0.0, 0.55, 0.0])
resid, jac = cbf._build(); r0 = resid(u_exp)
u_star, t_star, st, _ = cbf._solve(resid, jac, u_exp)
a = jac(u_exp)[0]; b = -r0[0] + a @ u_exp; Winv = cbf.Du ** 2
u_an = u_exp + (b - a @ u_exp) / (a @ (Winv * a)) * (Winv * a)
only_row0 = np.all(resid(u_an)[1:] >= -1e-9) and r0[0] < 0
record("3.4b CBF-QP equals analytic weighted projection onto the single active DC barrier", only_row0 and np.max(np.abs(u_star - u_an)) < 1e-6,
       f"|u*-u_analytic|_inf={np.max(np.abs(u_star - u_an)):.2e}, residual(u*)={resid(u_star)[0]:.1e}, status={st}")
pr.yf[2] = 0.70                                # unrecoverable in one step even with zero export
resid, jac = cbf._build(); u_star, t_star, st, _ = cbf._solve(resid, jac, u_exp)
lo = np.array([cons.P_min, cons.Q_min, cons.P_min, cons.Q_min]); hi = np.array([cons.P_max, cons.Q_max, cons.P_max, cons.Q_max])
a = jac(u_exp)[0]; u_best = np.where(a > 0, hi, lo); t_an = max(0.0, -(resid(u_exp)[0] + a @ (u_best - u_exp)))
record("3.4c CBF infeasible case: relaxation used, t* equals the analytic minimum (DC row)", st == 1 and abs(t_star - t_an) < 1e-6,
       f"status={st}, t*={t_star:.5f}, analytic={t_an:.5f}")

PASS 3.1 runner bit-identical to sim.run_closed_loop (Direct, Hard-limit; S1-S8) -- max |diff| = 0
PASS 3.2 nominal predictor exact on nominal plant (S1, one-step, k>=200) -- max one-step error = 2.22e-16 pu
PASS 3.3a RTA returns authority after exactly M_ret passing checks -- first s=1 at k=100 (predictor ready at k=1, M_ret=100)
PASS 3.3b RTA switches to fallback on the first failed check -- s before=1, s after forced failure=0
PASS 3.3c RTA monitor flags an unrecoverable command near the DC limit -- healthy state: pass=True (margin 0.0031); near-limit state: pass=False, first failing step=1, margin -0.0470
PASS 3.4a CBF passes a safe candidate unchanged -- dev=0.00e+00, status=0
PASS 3.4b CBF-QP equals analytic weighted projection onto the single active DC barrier -- |u*-u_analytic|_inf=5.55e-17, residual(u*)=0.0e+00, status=0
PASS 3.4c CBF infeasible case: relaxation used, t* equals the analytic minimum (DC row) -- status=1, t*=0.13912, analytic=0.13912


In [4]:
# 3.5 reproduction of the original CRI results with the unchanged code
ref = pd.read_csv(os.path.join(RESULTS_DIR, "results_strategy_comparison.csv"))
bench = Bench()
mism = []
for sname in SCENARIO_ORDER:
    for m, strat in [("Direct", "direct"), ("Hard-limit", "threshold"), ("EC-CRI", "cri")]:
        lg = bench.run(m, sname, SEED_REP); r = bench.metrics_A(lg, m)
        rr = ref[(ref.scenario == sname) & (ref.strategy == strat)].iloc[0]
        if r["N_FA"] != rr.N_FA or abs(r["E_V"] - rr.E_V) > 1e-12: mism.append((sname, m, r["N_FA"], rr.N_FA))
record("3.5a Protocol A: Direct/Hard-limit/EC-CRI reproduce results_strategy_comparison.csv (N_FA, E_V)", not mism, f"mismatches: {mism}")

ref_b = pd.read_csv(os.path.join(RESULTS_DIR, "results_dc_cri_loso_perscenario.csv"))
DC_FOLD = load_dc_fold_params(os.path.join(RESULTS_DIR, "results_dc_cri_loso_folds.csv"), crip)
mism = []
for sname in SCENARIO_ORDER:
    for seed in LOSO_EVAL_SEEDS[sname]:
        for m in ["Direct", "Hard-limit", "EC-CRI", "DC-CRI"]:
            lg = bench.run(m, sname, seed, crip_dc=DC_FOLD[sname]); r = bench.metrics_B(lg, m)
            rr = ref_b[(ref_b.held_out == sname) & (ref_b.method == m) & (ref_b.seed == seed)].iloc[0]
            if r["N_FA_H"] != rr.N_FA or abs(r["FAR"] - rr.FAR) > 1e-9 or abs(r["A_c"] - rr.A_c) > 1e-9:
                mism.append((sname, seed, m, r["N_FA_H"], rr.N_FA))
record("3.5b Protocol B: Direct/Hard-limit/EC-CRI/DC-CRI reproduce results_dc_cri_loso_perscenario.csv", not mism, f"mismatches: {mism}")
pd.DataFrame(checks).to_csv(os.path.join(RESULTS_DIR, "r1c7_validation_checks.csv"), index=False)
assert all(c["passed"] for c in checks), "A validation check failed -- stop and investigate"
print(f"All {len(checks)} validation checks passed ({time.time()-T_START:.0f} s elapsed)")

PASS 3.5a Protocol A: Direct/Hard-limit/EC-CRI reproduce results_strategy_comparison.csv (N_FA, E_V) -- mismatches: []
PASS 3.5b Protocol B: Direct/Hard-limit/EC-CRI/DC-CRI reproduce results_dc_cri_loso_perscenario.csv -- mismatches: []
All 10 validation checks passed (109 s elapsed)


## 4. Protocol A - EC protocol, instantaneous ground truth $z_k^{\rm unsafe}$

Seed 1 for every scenario (as in notebook 08), plus a 20-seed study for the two scenarios with measurement
noise (S6, S8; S1-S5 and S7 are deterministic). `Fallback-only` is a reference row (the fallback alone drives the
plant) used to separate supervisor effects from fallback limitations; it is not a method under test.

In [5]:
METHODS_A = ["Direct", "Hard-limit", "EC-CRI", "RTA", "CBF", "Fallback-only"]
rowsA = []
for sname in SCENARIO_ORDER:
    seeds = sorted(set([SEED_REP] + (SEEDS_A_NOISY if sname in NOISY else [])))
    for seed in seeds:
        for m in METHODS_A:
            lg = bench.run(m, sname, seed); r = bench.metrics_A(lg, m)
            r.update(scenario=sname, seed=seed, method=m); rowsA.append(r)
perA = pd.DataFrame(rowsA)
perA.to_csv(os.path.join(RESULTS_DIR, "r1c7_protocolA_perrun.csv"), index=False)
colsA = ["N_FA", "active_in_hazard_pct", "A_c", "safe_availability_pct", "n_violation_true", "max_exceedance_pu",
         "int_exceedance_pu_s", "E_V", "n_switch"]
repA = perA[perA.seed == SEED_REP].copy()
repA.to_csv(os.path.join(RESULTS_DIR, "r1c7_protocolA_perscenario_seed1.csv"), index=False)
aggA = repA.groupby("method", sort=False)[colsA].mean().reindex(METHODS_A)
aggA.to_csv(os.path.join(RESULTS_DIR, "r1c7_protocolA_aggregate_seed1.csv"))
noisyA = perA[perA.scenario.isin(sorted(NOISY))].groupby(["scenario", "method"], sort=False)[colsA].agg(["mean", "std"])
noisyA.to_csv(os.path.join(RESULTS_DIR, "r1c7_protocolA_noisy_20seed_summary.csv"))
print("Per-scenario N_FA (seed 1):"); print(repA.pivot(index="scenario", columns="method", values="N_FA")[METHODS_A])
print("\nPer-scenario true hard-limit violation samples (seed 1):"); print(repA.pivot(index="scenario", columns="method", values="n_violation_true")[METHODS_A])
print("\nPer-scenario A_c (seed 1; CBF = unmodified-command availability):"); print(repA.pivot(index="scenario", columns="method", values="A_c")[METHODS_A].round(3))
print("\nAggregate over S1-S8 (seed 1):"); print(aggA.round(4))
print("\nS6/S8 over 20 seeds (mean, std):"); print(noisyA[["N_FA", "n_violation_true", "A_c"]].round(3))
print(f"({time.time()-T_START:.0f} s elapsed)")

Per-scenario N_FA (seed 1):
method    Direct  Hard-limit  EC-CRI  RTA   CBF  Fallback-only
scenario                                                      
S1             0           0       0    0     0              0
S2             0           0       0    0     0              0
S3          1411         186       9   15   548              0
S4             0           0       0    0     0              0
S5          1570        1131     499  181   420              0
S6             0           0       0    0     0              0
S7          1101           2       5    3     3              0
S8          2501         835      19   57  1016              0

Per-scenario true hard-limit violation samples (seed 1):
method    Direct  Hard-limit  EC-CRI   RTA   CBF  Fallback-only
scenario                                                       
S1             0           0       0     0     0              0
S2             0           0       0   978     0            978
S3          1309        1014

## 5. Protocol B - LOSO protocol, finite-horizon label $z_k^{(H)}$ ($T_H=0.5$ s)

Evaluation seeds exactly as notebook 13 (seed 1; seeds 0-4 for S6, S8). DC-CRI uses the fold parameters calibrated
without the held-out scenario. RTA/CBF use one fixed configuration for all folds. Per-scenario values are seed means;
aggregates weight the eight scenarios equally (notebook 13 convention).

In [6]:
METHODS_B = ["Direct", "Hard-limit", "EC-CRI", "DC-CRI", "RTA", "CBF", "Fallback-only"]
rowsB = []
for sname in SCENARIO_ORDER:
    for seed in LOSO_EVAL_SEEDS[sname]:
        for m in METHODS_B:
            lg = bench.run(m, sname, seed, crip_dc=DC_FOLD[sname]); r = bench.metrics_B(lg, m)
            r.update(held_out=sname, seed=seed, method=m); rowsB.append(r)
perB = pd.DataFrame(rowsB)
perB.to_csv(os.path.join(RESULTS_DIR, "r1c7_protocolB_perrun.csv"), index=False)
colsB = ["N_FA_H", "FAR", "FDR", "A_c", "E_V", "n_violation_true", "max_exceedance_pu", "int_exceedance_pu_s", "n_switch"]
extra = ["rta_n_interventions", "rta_monitor_alarm_pct", "cbf_filtered_pct", "cbf_mean_dev_inf", "cbf_relaxed_pct",
         "cbf_solver_fail_pct", "sup_time_mean_ms", "sup_time_max_ms"]
psB = perB.groupby(["held_out", "method"], sort=False)[colsB + extra].mean().reset_index()
psB.to_csv(os.path.join(RESULTS_DIR, "r1c7_protocolB_perscenario.csv"), index=False)
aggB = psB.groupby("method", sort=False)[colsB].agg(["mean", "std"]).reindex(METHODS_B)
aggB.to_csv(os.path.join(RESULTS_DIR, "r1c7_protocolB_aggregate.csv"))
print(psB.pivot(index="held_out", columns="method", values="FAR")[METHODS_B].round(3))
print(psB.pivot(index="held_out", columns="method", values="A_c")[METHODS_B].round(3))
print(psB.pivot(index="held_out", columns="method", values="n_violation_true")[METHODS_B].round(1))
print("\nAggregate (mean over 8 equally weighted held-out scenarios):"); print(aggB.xs("mean", axis=1, level=1).round(4))
print("\nBaseline diagnostics per scenario:"); print(psB[psB.method.isin(["RTA", "CBF"])][["held_out", "method"] + extra].round(3).to_string(index=False))
print(f"({time.time()-T_START:.0f} s elapsed)")

method    Direct  Hard-limit  EC-CRI  DC-CRI    RTA    CBF  Fallback-only
held_out                                                                 
S1           0.0       0.000   0.000   0.000  0.000  0.000            0.0
S2           0.0       0.000   0.000   0.027  0.000  0.000            0.0
S3           1.0       0.223   0.000   0.000  0.033  0.881            0.0
S4           0.0       0.000   0.000   0.000  0.000  0.000            0.0
S5           1.0       0.705   0.241   0.678  0.103  0.962            0.0
S6           0.0       0.000   0.000   0.000  0.000  0.000            0.0
S7           1.0       0.023   0.026   0.032  0.024  0.024            0.0
S8           1.0       0.189   0.000   1.000  0.001  0.245            0.0
method    Direct  Hard-limit  EC-CRI  DC-CRI    RTA    CBF  Fallback-only
held_out                                                                 
S1           1.0       1.000   0.999   1.000  0.945  1.000            0.0
S2           1.0       1.000   0.999  

## 6. Sensitivity to the a-priori baseline settings (reported, not used for selection)

The development-set selection changed the RTA dwell (25 -> 100 steps) and the CBF decay rate (0.2 -> 1.0).
The a-priori values are evaluated under Protocol B so the reader can see how much the conclusions depend on them.

In [7]:
variants = {"RTA (M_ret=25, a priori)": ("RTA", Bench(rta_prm=bl.RTAParams(return_dwell_steps=25))),
            "CBF (gamma=0.2, a priori)": ("CBF", Bench(cbf_prm=bl.CBFParams(gamma=0.2)))}
rowsS = []
for label, (m, bvar) in variants.items():
    for sname in SCENARIO_ORDER:
        for seed in LOSO_EVAL_SEEDS[sname]:
            lg = bvar.run(m, sname, seed); r = bvar.metrics_B(lg, m); r.update(held_out=sname, seed=seed, method=label); rowsS.append(r)
perS = pd.DataFrame(rowsS)
psS = perS.groupby(["held_out", "method"], sort=False)[colsB].mean().reset_index()
psS.to_csv(os.path.join(RESULTS_DIR, "r1c7_sensitivity_protocolB_perscenario.csv"), index=False)
aggS = psS.groupby("method", sort=False)[colsB].mean()
aggS = pd.concat([aggB.xs("mean", axis=1, level=1).loc[["RTA", "CBF"]].rename(index={"RTA": "RTA (M_ret=100, selected)", "CBF": "CBF (gamma=1.0, selected)"}), aggS])
aggS.to_csv(os.path.join(RESULTS_DIR, "r1c7_sensitivity_protocolB_aggregate.csv"))
print(aggS.round(4)); print(f"({time.time()-T_START:.0f} s elapsed)")

                           N_FA_H     FAR     FDR     A_c     E_V  n_violation_true  max_exceedance_pu  int_exceedance_pu_s  n_switch
method                                                                                                                               
RTA (M_ret=100, selected)   7.925  0.0201  0.3973  0.4736  0.0514           537.700             0.0740               1.3603     9.350
CBF (gamma=1.0, selected)  80.100  0.2640  0.1394  0.7484  0.0428           282.600             0.0831               1.5650    40.425
RTA (M_ret=25, a priori)   19.275  0.0478  0.2371  0.5891  0.0504           538.825             0.0739               1.3606    22.225
CBF (gamma=0.2, a priori)  14.225  0.0092  0.2231  0.6774  0.0434           296.900             0.0803               1.5916    10.075
(619 s elapsed)


## 7. Summary tables (LaTeX, generated directly from the saved CSVs)

In [8]:
def f3(x): return f"{x:.3f}"
def f1(x): return f"{x:.1f}"
# Table A: Protocol B aggregate (main comparison)
lab = {"Direct": "Direct activation", "Hard-limit": "Hard-limit supervisor", "EC-CRI": "EC-CRI", "DC-CRI": "DC-CRI",
       "RTA": "Simplex RTA", "CBF": "DT-CBF filter$^{\\dagger}$", "Fallback-only": "Fallback only (ref.)"}
mB, sB = aggB.xs("mean", axis=1, level=1), aggB.xs("std", axis=1, level=1)
lines = [r"\begin{tabular}{lcccccc}", r"\toprule",
         r"Method & $N_{\mathrm{FA}}^{(H)}$ & FAR & FDR & $A_c$ & $N_{\mathrm{viol}}$ & $\int e^{+}\,dt$ [pu$\cdot$s] \\", r"\midrule"]
for m in METHODS_B:
    lines.append(f"{lab[m]} & {f1(mB.loc[m,'N_FA_H'])} & {f3(mB.loc[m,'FAR'])}$\\pm${f3(sB.loc[m,'FAR'])} & {f3(mB.loc[m,'FDR'])} & "
                 f"{f3(mB.loc[m,'A_c'])}$\\pm${f3(sB.loc[m,'A_c'])} & {f1(mB.loc[m,'n_violation_true'])} & {mB.loc[m,'int_exceedance_pu_s']:.4f} \\\\")
lines += [r"\bottomrule", r"\end{tabular}"]
open(os.path.join(RESULTS_DIR, "r1c7_table_protocolB_aggregate.tex"), "w").write("\n".join(lines))
# Table B: per-scenario FAR / A_c / violations (Protocol B)
meths = ["EC-CRI", "DC-CRI", "RTA", "CBF"]
lines = [r"\begin{tabular}{l" + "ccc" * len(meths) + "}", r"\toprule",
         "Held-out & " + " & ".join([rf"\multicolumn{{3}}{{c}}{{{lab[m]}}}" for m in meths]) + r" \\",
         " & " + " & ".join(["FAR & $A_c$ & $N_{\\mathrm{viol}}$"] * len(meths)) + r" \\", r"\midrule"]
for sname in SCENARIO_ORDER:
    cells = []
    for m in meths:
        r = psB[(psB.held_out == sname) & (psB.method == m)].iloc[0]
        cells += [f3(r.FAR), f3(r.A_c), f1(r.n_violation_true)]
    lines.append(sname + " & " + " & ".join(cells) + r" \\")
lines += [r"\bottomrule", r"\end{tabular}"]
open(os.path.join(RESULTS_DIR, "r1c7_table_protocolB_perscenario.tex"), "w").write("\n".join(lines))
# Table C: Protocol A aggregate (seed 1)
lines = [r"\begin{tabular}{lcccccc}", r"\toprule",
         r"Method & $N_{\mathrm{FA}}$ & Active-in-hazard [\%] & $A_c$ & Safe avail. [\%] & $N_{\mathrm{viol}}$ & $E_V$ \\", r"\midrule"]
for m in METHODS_A:
    r = aggA.loc[m]
    lines.append(f"{lab[m]} & {f1(r.N_FA)} & {f1(r.active_in_hazard_pct)} & {f3(r.A_c)} & {f1(r.safe_availability_pct)} & {f1(r.n_violation_true)} & {r.E_V:.4f} \\\\")
lines += [r"\bottomrule", r"\end{tabular}"]
open(os.path.join(RESULTS_DIR, "r1c7_table_protocolA_aggregate.tex"), "w").write("\n".join(lines))
print(open(os.path.join(RESULTS_DIR, "r1c7_table_protocolB_aggregate.tex")).read())

\begin{tabular}{lcccccc}
\toprule
Method & $N_{\mathrm{FA}}^{(H)}$ & FAR & FDR & $A_c$ & $N_{\mathrm{viol}}$ & $\int e^{+}\,dt$ [pu$\cdot$s] \\
\midrule
Direct activation & 794.9 & 0.500$\pm$0.535 & 0.000 & 1.000$\pm$0.000 & 694.7 & 2.9443 \\
Hard-limit supervisor & 222.1 & 0.142$\pm$0.245 & 0.022 & 0.756$\pm$0.287 & 520.9 & 1.3199 \\
EC-CRI & 6.1 & 0.033$\pm$0.084 & 0.247 & 0.683$\pm$0.377 & 323.9 & 1.3027 \\
DC-CRI & 374.3 & 0.217$\pm$0.394 & 0.054 & 0.806$\pm$0.225 & 518.0 & 1.4494 \\
Simplex RTA & 7.9 & 0.020$\pm$0.036 & 0.397 & 0.474$\pm$0.347 & 537.7 & 1.3603 \\
DT-CBF filter$^{\dagger}$ & 80.1 & 0.264$\pm$0.415 & 0.139 & 0.748$\pm$0.299 & 282.6 & 1.5650 \\
Fallback only (ref.) & 0.0 & 0.000$\pm$0.000 & 1.000 & 0.000$\pm$0.000 & 445.1 & 1.3604 \\
\bottomrule
\end{tabular}


In [9]:
meta["validation_checks"] = checks
meta["runtime_s"] = round(time.time() - T_START, 1)
meta["outputs"] = sorted(f for f in os.listdir(RESULTS_DIR) if f.startswith("r1c7_"))
json.dump(meta, open(os.path.join(RESULTS_DIR, "r1c7_run_metadata.json"), "w"), indent=1, default=str)
print(f"Done in {meta['runtime_s']} s. Outputs: {meta['outputs']}")

Done in 619.1 s. Outputs: ['r1c7_devset_selection.csv', 'r1c7_protocolA_aggregate_seed1.csv', 'r1c7_protocolA_noisy_20seed_summary.csv', 'r1c7_protocolA_perrun.csv', 'r1c7_protocolA_perscenario_seed1.csv', 'r1c7_protocolB_aggregate.csv', 'r1c7_protocolB_perrun.csv', 'r1c7_protocolB_perscenario.csv', 'r1c7_sensitivity_protocolB_aggregate.csv', 'r1c7_sensitivity_protocolB_perscenario.csv', 'r1c7_table_protocolA_aggregate.tex', 'r1c7_table_protocolB_aggregate.tex', 'r1c7_table_protocolB_perscenario.tex', 'r1c7_validation_checks.csv']


## 8. Findings

All values below are read from the CSV files written above (Protocol B: equal-weight mean over the eight held-out
scenarios; Protocol A: mean over S1-S8, seed 1).

| Method | FAR (B) | FDR (B) | $A_c$ (B) | $N_{\rm viol}$ (B) | $N_{FA}$ (A) | $A_c$ (A) |
|---|---|---|---|---|---|---|
| Direct | 0.500 | 0.000 | 1.000 | 694.7 | 822.9 | 1.000 |
| Hard-limit | 0.142 | 0.022 | 0.756 | 520.9 | 269.3 | 0.755 |
| EC-CRI | 0.033 | 0.247 | 0.683 | 323.9 | 66.5 | 0.684 |
| DC-CRI | 0.217 | 0.054 | 0.806 | 518.0 | - | - |
| Simplex RTA | 0.020 | 0.397 | 0.474 | 537.7 | 32.0 | 0.461 |
| DT-CBF filter | 0.264 | 0.139 | 0.748 | 282.6 | 248.4 | 0.749 |
| Fallback only (reference) | 0.000 | 1.000 | 0.000 | 445.1 | 0.0 | 0.000 |

* **No method dominates; the comparison exposes a safety-availability trade-off.** Simplex RTA attains the lowest
  false-activation rate (FAR 0.020 vs. 0.033 for EC-CRI) but is the most conservative supervisor (availability 0.474,
  FDR 0.397). EC-CRI keeps a low FAR at higher availability (0.683). The CBF filter keeps the candidate unmodified
  most often among the safety-supervised methods (0.748) but at the highest FAR (0.264), and DC-CRI reaches the highest
  availability among the supervised methods (0.806) at FAR 0.217.
* **The CBF filter yields the fewest true hard-limit violation samples** (282.6 vs. 323.9 for EC-CRI), but the
  largest maximum and integrated exceedance (0.083 pu, 1.565 pu s), because its guarantees rest on the nominal model.
  In S7/S8 (converter outage, model mismatch) it needs the feasibility relaxation in 55 % / 40 % of the samples,
  with a worst-case filter time of about 50 ms.
* **Most remaining violations are not caused by the supervisor.** In S7 every method, including the fallback alone,
  produces the same 1100 violation samples, and in S8 the fallback alone produces about 1483; the fallback controller
  itself does not keep the plant admissible in these scenarios, so no supervisor that hands authority back to it can.
* All ten validation checks (Sec. 3) pass, including bit-identical reproduction of the stored Direct/Hard-limit/EC-CRI
  and DC-CRI results, so the new runner introduces no change to existing results.
* **The baseline results depend on their tuning (Sec. 6).** The settings were selected on the development set by
  hard-limit violation count. With the a-priori CBF decay rate $\gamma=0.2$ instead of the selected $\gamma=1$, the CBF
  filter reaches FAR 0.009 at availability 0.677, i.e. a lower FAR than EC-CRI (0.033) at comparable availability
  (0.683), with slightly more violation samples (296.9 vs. 282.6). With the a-priori RTA dwell of 25 steps, RTA moves to
  FAR 0.048 at availability 0.589. The ranking between EC-CRI and the baselines is therefore setting dependent.

**Scope.** Results are for the simulated hybrid AC/DC microgrid; RTA and CBF are reference implementations built on a
shared nominal-model predictor, not tuned commercial products.
